### Problem Statement
* Sometimes, in a project, the client demands to implement <b>SCD2</b>, in order to keep the <b>historical data</b> in a <b>Silver Delta</b> table in <b>Databricks</b>, whereas the corresponding source <b>Bronze Delta</b> table does not have any column having uniquely identifiable record, or, any audit column.
* Also, in most of the cases, that source <b>Bronze Delta</b> table is truncated and loaded each day.
<br>So, it is not possible to find out which record gets inserted, updated, or deleted.

### Create a Bronze Delta Table with Change-Data-Feed Feature Enabled

In [0]:
%sql
DROP TABLE IF EXISTS oc_catalog.oc_schema.scd2_use_case1_bronze;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS oc_catalog.oc_schema.scd2_use_case1_bronze (
    user_email_id STRING NOT NULL COMMENT 'Mail ID of the user',
    user_id STRING COMMENT 'ID of the user',
    country STRING COMMENT 'Country of the user',
    city STRING COMMENT 'City of the user'
)
TBLPROPERTIES (
    'comment' = 'This is a bronze table',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.enableChangeDataFeed' = 'true',
    'delta.enableDeletionVectors' = 'true'
)

### Insert Data for the First Time into the Bronze Delta Table

In [0]:
%sql
INSERT INTO oc_catalog.oc_schema.scd2_use_case1_bronze (user_email_id, user_id, country, city) VALUES
('oindrila.chakraborty88@gmail.com', '703390592', 'India', 'Kolkata'),
('soumyajyoti.bagchi@gmail.com', '426248', 'India', 'Kolkata')

### Display the Data of the Bronze Delta Table

In [0]:
%sql
SELECT * FROM oc_catalog.oc_schema.scd2_use_case1_bronze

### Read All the Changes Made to Data After Performing Operations on a Delta Table Using "Change Data Feed" Feature
* To read all the changes made to a <b>Batch Delta</b> table, the <b>table_changes ()</b> function is used.
  * To fetch all the changes made to a <b>Batch Delta</b> table betwen two specific <b>versions</b>, the following command is used -
  <br><b>SELECT * FROM table_changes('`<tableName>`', `<starting-version>`, `<ending-version>`)</b>
  <br>Both the <b>starting</b> and <b>ending versions</b> are inclusive.
  * To fetch all the changes made to a <b>Batch Delta</b> table from a <b>starting version</b> to the <b>latest version</b>, only the <b>starting version</b> needs to be specified as shown in the following command -
  <br><b>SELECT * FROM table_changes('`<tableName>`', `<starting-version>`)</b>
  * To fetch all the changes made to a <b>Batch Delta</b> table in the <b>latest version</b>, the following command is used -
  <br><b>SELECT * FROM table_changes('`<tableName>`', `<latest-version>`)</b>
  <br>This is similar to fetching the changes from a <b>starting version</b> to the <b>latest version</b>, with the only difference is that, both the <b>starting version</b> and the <b>latest version</b> is same.

### Display All the Changes Made to the Bronze Delta Table 'oc_catalog.oc_schema.scd2_use_case1_bronze' from the Starting Version to the Latest Version

In [0]:
%sql
SELECT * FROM table_changes('oc_catalog.oc_schema.scd2_use_case1_bronze', 0)

### Truncate the Data of the Bronze Delta Table for the First Time

In [0]:
%sql
TRUNCATE TABLE oc_catalog.oc_schema.scd2_use_case1_bronze

### Insert Data for the Second Time into the Bronze Delta Table, After First Truncate

In [0]:
%sql
INSERT INTO oc_catalog.oc_schema.scd2_use_case1_bronze (user_email_id, user_id, country, city) VALUES
('oindrila.chakraborty88@gmail.com', '703390592', 'India', 'Kolkata'),
('soumyajyoti.bagchi@gmail.com', '426248', 'India', 'Kolkata'),
('debajyoti.bagchi55@gmail.com', '123097', 'India', 'Kolkata'),
('dola.bagchi60@gmail.com', '100934', 'India', 'Kolkata')

### Display All the Changes Made to the Bronze Delta Table 'oc_catalog.oc_schema.scd2_use_case1_bronze' from the Starting Version to the Latest Version

In [0]:
%sql
SELECT * FROM table_changes('oc_catalog.oc_schema.scd2_use_case1_bronze', 0)

* It can be seen that although, the users with the mail ids <b>oindrila.chakraborty88@gmail.com</b>, and, <b>soumyajyoti.bagchi@gmail.com</b> were already inserted in the <b>version</b> of <b>1</b> into the <b>Delta</b> table, still in the <b>version</b> of <b>3</b>, again the users are specified as <b>insert</b> in the column <b>_change_type</b> in the changes returned by the function <b>table_changes ()</b> for the <b>Delta</b> table.

### Truncate the Data of the Bronze Delta Table for the Second Time

In [0]:
%sql
TRUNCATE TABLE oc_catalog.oc_schema.scd2_use_case1_bronze

### Insert Data for the Third Time into the Bronze Delta Table, After Second Truncate

In [0]:
%sql
INSERT INTO oc_catalog.oc_schema.scd2_use_case1_bronze (user_email_id, user_id, country, city) VALUES
('soumyajyoti.bagchi@gmail.com', '426248', 'India', 'Kolkata'),
('debajyoti.bagchi55@gmail.com', '123097', 'India', 'Kolkata'),
('dola.bagchi60@gmail.com', '100934', 'India', 'Kolkata'),
('oindrila.chakraborty88@gmail.com', '703390592', 'Japan', 'Tokyo'),
('abhirup.chakraborty88@gmail.com', '560098432', 'USA', 'New Jersey')

### Display All the Changes Made to the Bronze Delta Table 'oc_catalog.oc_schema.scd2_use_case1_bronze' from the Starting Version to the Latest Version

In [0]:
%sql
SELECT * FROM table_changes('oc_catalog.oc_schema.scd2_use_case1_bronze', 0)

### Fetch the Latest Version of the Bronze Delta Table 'oc_catalog.oc_schema.scd2_use_case1_bronze'

In [0]:
latest_version = spark.sql("""
SELECT MAX(version) AS latest_version
FROM (DESCRIBE HISTORY oc_catalog.oc_schema.scd2_use_case1_bronze)
""").collect()[0][0]
print("The Latest Version of the Bronze Delta Table 'oc_catalog.oc_schema.scd2_use_case1_bronze' is: ", latest_version)

### Display All the Changes Made to the Bronze Delta Table 'oc_catalog.oc_schema.scd2_use_case1_bronze' in the Latest Version

In [0]:
spark.sql(f"""
SELECT * FROM table_changes('oc_catalog.oc_schema.scd2_use_case1_bronze', {latest_version})
""").display()

### The Problem of Truncate and Load Strategy When Data Gets Updated
* So, it can be seen that in the <b>version 1</b>, and, in the <b>version 3</b>, the value of the columns <b>country</b>, and, <b>city</b> were same for the user with the email id <b>oindrila.chakraborty88@gmail.com</b>.
* But, the value of the columns <b>country</b>, and, <b>city</b> got changed for the user with the email id <b>oindrila.chakraborty88@gmail.com</b> in the <b>version 5</b>.
<br>But, as the <b>Delta</b> table is truncated each time before loading with the data, whatever record is updated is not getting logged in the <b>change data feed</b> as all the records are getting inserted each time.
* So, it is impossible to distinguish which records are getting inserted, which records are getting deleted, and, which records are getting updated using the <b>table_changes ()</b> function for a <b>Delta</b> table that gets truncated each time before loading with data.
* In the <b>version 5</b>, only the newly inserted record is for the user with the email id <b>abhirup.chakraborty88@gmail.com</b>, where as, the user with the email id <b>oindrila.chakraborty88@gmail.com</b> has the value of the columns <b>country</b>, and, <b>city</b> updated, and, the rest of the users do not have any changes.
<br>Still, all the users are specified as <b>insert</b> in the column <b>_change_type</b> in the changes returned by the function <b>table_changes ()</b> for the <b>Delta</b> table.

### Create a Silver Delta Table with Change-Data-Feed Feature Enabled

In [0]:
%sql
DROP TABLE IF EXISTS oc_catalog.oc_schema.scd2_use_case1_silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS oc_catalog.oc_schema.scd2_use_case1_silver (
    user_email_id STRING NOT NULL COMMENT 'Mail ID of the user',
    user_id STRING COMMENT 'ID of the user',
    country STRING COMMENT 'Country of the user',
    city STRING COMMENT 'City of the user',
    eff_start_tms TIMESTAMP COMMENT 'The start timestamp of a record',
    eff_end_tms TIMESTAMP COMMENT 'The end timestamp of a record',
    active_flag STRING NOT NULL COMMENT 'Denotes if the record is active'
)
TBLPROPERTIES (
    'comment' = 'This is a silver table',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.enableChangeDataFeed' = 'true',
    'delta.enableDeletionVectors' = 'true'
)

### Create the Dataset that Has Only Records to be Updated

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW vw_update_dataset
AS
SELECT DISTINCT         target.user_email_id AS user_email_id,
                        target.user_id AS user_id,
                        target.country AS country,
                        target.city AS city,
                        target.eff_start_tms AS eff_start_tms
FROM                    oc_catalog.oc_schema.scd2_use_case1_bronze source
INNER JOIN              oc_catalog.oc_schema.scd2_use_case1_silver target
ON                      target.user_email_id = source.user_email_id
AND                     target.user_id = source.user_id
AND                     target.active_flag = 'Y'
AND                     (target.country <> source.country OR target.city <> source.city)

In [0]:
%sql
SELECT * FROM vw_update_dataset;

### Create the Dataset that Has Only Records to be Newly Inserted (Both Fresh and Insert Counterpart of Updated Data)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW vw_insert_dataset
AS
-- Fetch Insert Counterpart of Updated Data
SELECT DISTINCT         source.user_email_id AS user_email_id,
                        source.user_id AS user_id,
                        source.country AS country,
                        source.city AS city
FROM                    oc_catalog.oc_schema.scd2_use_case1_bronze source
INNER JOIN              oc_catalog.oc_schema.scd2_use_case1_silver AS target
ON                      target.user_email_id = source.user_email_id
AND                     target.user_id = source.user_id
AND                     active_flag = 'Y'
AND                     (target.country <> source.country OR target.city <> source.city)

UNION ALL

-- Fetch Newly Inserted Data
SELECT DISTINCT         source.user_email_id AS user_email_id,
                        source.user_id AS user_id,
                        source.country AS country,
                        source.city AS city
FROM                    oc_catalog.oc_schema.scd2_use_case1_bronze source
LEFT ANTI JOIN          oc_catalog.oc_schema.scd2_use_case1_silver target
ON                      target.user_email_id = source.user_email_id
AND                     target.user_id = source.user_id
AND                     target.active_flag = 'Y'

In [0]:
%sql
SELECT * FROM vw_insert_dataset;

### Create the Dataset that Has Only Records to be Deleted, i.e., Not Present in the Source Bronze Table Anymore

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW vw_delete_dataset
AS
WITH CTE_Data_To_Delete AS
(
    SELECT                  user_email_id AS user_email_id,
                            user_id AS user_id,
                            country AS country,
                            city AS city,
                            eff_start_tms AS eff_start_tms,
                            ROW_NUMBER() OVER (PARTITION BY user_email_id, user_id ORDER BY eff_start_tms DESC) AS row_num
    FROM                    oc_catalog.oc_schema.scd2_use_case1_silver
    WHERE                   active_flag = 'Y'
) 
SELECT                  target.user_email_id AS user_email_id,
                        target.user_id AS user_id,
                        target.country AS country,
                        target.city AS city,
                        target.eff_start_tms
FROM                    CTE_Data_To_Delete AS target
LEFT ANTI JOIN          oc_catalog.oc_schema.scd2_use_case1_bronze source
ON                      target.user_email_id = source.user_email_id
AND                     target.user_id = source.user_id
WHERE row_num = 1;

In [0]:
%sql
SELECT * FROM vw_delete_dataset;

### Create Final Source Dataset that Has All - Records to be Updated, As Well As Records to be Newly Inserted (Both Fresh and Insert Counterpart of Updated Data), And, Records to be Deleted, Which are Not Present in Source Bronze Table Anymore

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW vw_final_dataset
AS
SELECT DISTINCT         *,
                        CURRENT_TIMESTAMP () AS eff_end_tms,
                        'N' AS active_flag
FROM                    vw_update_dataset

UNION ALL

SELECT DISTINCT         *,
                        CURRENT_TIMESTAMP () AS eff_start_tms,
                        NULL AS eff_end_tms,
                        'Y' AS active_flag
FROM                    vw_insert_dataset

UNION ALL

SELECT DISTINCT         *,
                        CURRENT_TIMESTAMP () AS eff_end_tms,
                        'N' AS active_flag
FROM                    vw_delete_dataset

In [0]:
%sql
SELECT * FROM vw_final_dataset;

### Perform Upsert

In [0]:
spark.sql(f"""
MERGE INTO oc_catalog.oc_schema.scd2_use_case1_silver AS target
USING vw_final_dataset AS source 
    ON target.user_email_id = source.user_email_id
    AND target.user_id = source.user_id
    AND source.active_flag = 'N'
    AND target.eff_start_tms = source.eff_start_tms
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

### Verify the Data of the Silver Table 'oc_catalog.oc_schema.scd2_use_case1_silver'

In [0]:
%sql
SELECT * FROM oc_catalog.oc_schema.scd2_use_case1_silver;

### Truncate the Data of the Bronze Delta Table for the Third Time

In [0]:
%sql
TRUNCATE TABLE oc_catalog.oc_schema.scd2_use_case1_bronze

### Insert Data for the Fourth Time into the Bronze Delta Table, After Third Truncate

In [0]:
%sql
INSERT INTO oc_catalog.oc_schema.scd2_use_case1_bronze (user_email_id, user_id, country, city) VALUES
('soumyajyoti.bagchi@gmail.com', '426248', 'India', 'Kolkata'),
('debajyoti.bagchi55@gmail.com', '123097', 'India', 'Kolkata'),
('dola.bagchi60@gmail.com', '100934', 'India', 'Kolkata'),
('oindrila.chakraborty88@gmail.com', '703390592', 'India', 'Kolkata'),
('ankan.lahiri93@gmail.com', '302387', 'India', 'Gurgaon')

* It can be seen that after the third truncate, the user with the mail id <b>ankan.lahiri93@gmail.com</b> is newly inserted, and, the user with the mail id <b>abhirup.chakraborty88@gmail.com</b> is deleted in the <b>version</b> of <b>7</b> of the <b>Delta</b> table.
* Also, the value of the columns <b>country</b> and <b>city</b> of the user with the mail id <b>oindrila.chakraborty88@gmail.com</b> are updated.
* All other users remain same.

### Create Final Dataset Again and Perform Upsert
* Now, again create the dataset for updates only, to be newly inserted (both fresh and insert counterpart of updated data), as well as for the records that need to be deleted, which are not present in the source <b>Delta</b> table.
* Perform the <b>merge</b> operation, and, check the data present in the <b>Delta</b> table.

### Verify the Data of the Silver Table 'oc_catalog.oc_schema.scd2_use_case1_silver'

In [0]:
%sql
SELECT * FROM oc_catalog.oc_schema.scd2_use_case1_silver;

### Truncate the Data of the Bronze Delta Table for the Fourth Time

In [0]:
%sql
TRUNCATE TABLE oc_catalog.oc_schema.scd2_use_case1_bronze

### Insert Data for the Fifth Time into the Bronze Delta Table, After Fourth Truncate

In [0]:
%sql
INSERT INTO oc_catalog.oc_schema.scd2_use_case1_bronze (user_email_id, user_id, country, city) VALUES
('soumyajyoti.bagchi@gmail.com', '426248', 'India', 'Hyderabad'),
('debajyoti.bagchi55@gmail.com', '123097', 'India', 'Kolkata'),
('dola.bagchi60@gmail.com', '100934', 'India', 'Kolkata'),
('oindrila.chakraborty88@gmail.com', '703390592', 'Japan', 'Okinawa'),
('abhirup.chakraborty88@gmail.com', '560098432', 'USA', 'New Jersey')

* It can be seen that after the fourth truncate, the user with the mail id <b>abhirup.chakraborty88@gmail.com</b> is newly inserted after the deletion in the last version, and, the user with the mail id <b>ankan.lahiri93@gmail.com</b> is deleted in the <b>version</b> of <b>9</b> of the <b>Delta</b> table.
* Also, the values of the columns <b>country</b> and <b>city</b> of the users with the mail ids <b>soumyajyoti.bagchi@gmail.com</b> and <b>oindrila.chakraborty88@gmail.com</b> are updated.
* All other users remain same.

### Create Final Dataset Again and Perform Upsert
* Now, again create the dataset for updates only, to be newly inserted (both fresh and insert counterpart of updated data), as well as for the records that need to be deleted, which are not present in the source Delta table.
* Perform the merge operation, and, check the data present in the Delta table.

### Verify the Data of the Silver Table 'oc_catalog.oc_schema.scd2_use_case1_silver'

In [0]:
%sql
SELECT * FROM oc_catalog.oc_schema.scd2_use_case1_silver;

### Truncate the Data of the Bronze Delta Table for the Fifth Time

In [0]:
%sql
TRUNCATE TABLE oc_catalog.oc_schema.scd2_use_case1_bronze

### Insert Data for the Sixth Time into the Bronze Delta Table, After Fifth Truncate

In [0]:
%sql
INSERT INTO oc_catalog.oc_schema.scd2_use_case1_bronze (user_email_id, user_id, country, city) VALUES
('soumyajyoti.bagchi@gmail.com', '426248', 'India', 'Kolkata'),
('debajyoti.bagchi55@gmail.com', '123097', 'India', 'Kolkata'),
('dola.bagchi60@gmail.com', '100934', 'India', 'Kolkata'),
('oindrila.chakraborty88@gmail.com', '703390592', 'India', 'Kolkata')

* It can be seen that after the fifth truncate, the user with the mail id <b>abhirup.chakraborty88@gmail.com</b> is deleted in the latest <b>version</b> of the <b>Delta</b> table.
* Also, the values of the columns <b>country</b> and <b>city</b> of the users with the mail ids <b>soumyajyoti.bagchi@gmail.com</b> and <b>oindrila.chakraborty88@gmail.com</b> are updated.
* All other users remain same.

### Verify the Data of the Silver Table 'oc_catalog.oc_schema.scd2_use_case1_silver'

In [0]:
%sql
SELECT * FROM oc_catalog.oc_schema.scd2_use_case1_silver;

### Truncate the Data of the Bronze Delta Table for the Sixth Time

In [0]:
%sql
TRUNCATE TABLE oc_catalog.oc_schema.scd2_use_case1_bronze

### Insert Data for the Seventh Time into the Bronze Delta Table, After Sixth Truncate

In [0]:
%sql
INSERT INTO oc_catalog.oc_schema.scd2_use_case1_bronze (user_email_id, user_id, country, city) VALUES
('soumyajyoti.bagchi@gmail.com', '426248', 'India', 'Kolkata'),
('debajyoti.bagchi55@gmail.com', '123097', 'India', 'Kolkata'),
('dola.bagchi60@gmail.com', '100934', 'India', 'Kolkata'),
('oindrila.chakraborty88@gmail.com', '703390592', 'India', 'Kolkata'),
('abhirup.chakraborty88@gmail.com', '560098432', 'USA', 'New Jersey')

* It can be seen that after the sixth truncate, the user with the mail id <b>abhirup.chakraborty88@gmail.com</b> is re-inserted in the latest <b>version</b> of the <b>Delta</b> table.
* All other users remain same.

### Verify the Data of the Silver Table 'oc_catalog.oc_schema.scd2_use_case1_silver'

In [0]:
%sql
SELECT * FROM oc_catalog.oc_schema.scd2_use_case1_silver;